# NodPT Prompt Generators

This notebook covers the two prompt-generation scripts in `AI/src/`:

1. **`run.py`** — Send structured prompts to a local Ollama model and receive JSON-validated responses for any NodPT node type.
2. **`generate_vllm_samples.py`** — Generate coding and book-writing training samples via a vLLM server, with randomised topics and concurrent async batching.

---

## Architecture Overview

```
AI/src/
├── run.py                    # Prompt runner — Ollama /api/generate (structured JSON output)
├── generate_vllm_samples.py  # Batch sample generator — vLLM /v1/chat/completions
├── {Director,Manager,Supervisor,Agent}/
│   ├── format.json           # Ollama JSON Schema for structured output
│   └── prompts/
│       └── sample.txt        # Example prompt for this node type
└── vllm-samples/             # Generated JSONL output from generate_vllm_samples.py
```

---

## Prerequisites

```bash
cd AI/src
pip install -r requirements.txt
```

- **`run.py`** requires [Ollama](https://ollama.ai/) running locally (`ollama serve`).
- **`generate_vllm_samples.py`** requires a [vLLM](https://github.com/vllm-project/vllm) server on port 8001.

---
## Part 1 — Ollama Structured Prompt Runner (`run.py`)

**Script:** `AI/src/run.py`

`run.py` is the main CLI for interacting with a locally running Ollama model through NodPT's node-type system. It:
1. Loads the **JSON Schema** for the selected node type (`format.json`).
2. Loads a **prompt text** from a `.txt` file in `<NodeType>/prompts/` (or from `--prompt-text`).
3. Sends a request to Ollama's `/api/generate` endpoint with the schema attached as the `format` field, which constrains Ollama to produce valid structured JSON.
4. Displays the parsed JSON response.

### 1.1 Imports and Configuration

`requests` is used for synchronous HTTP calls since run.py is a single-shot CLI tool (no concurrency needed). All defaults are overridable via CLI arguments.

In [ ]:
import argparse
import json
import os
import sys

import requests

# ── Base directory: AI/src/ ───────────────────────────────────────────────────
# Adjust if running the notebook from a different working directory.
BASE_DIR = os.path.abspath("AI/src")

# ── Supported node types (capitalised to match folder names) ─────────────────
NODE_TYPES = ["Director", "Manager", "Supervisor", "Agent"]

# ── Ollama defaults ───────────────────────────────────────────────────────────
DEFAULT_OLLAMA_URL = "http://localhost:11434"  # Standard Ollama port
DEFAULT_MODEL      = "llama3.1:8b"             # Model tag as shown by `ollama list`

print("Configuration loaded.")
print(f"Node types: {NODE_TYPES}")
print(f"Ollama URL: {DEFAULT_OLLAMA_URL}")
print(f"Default model: {DEFAULT_MODEL}")

### 1.2 Schema and Prompt Loaders

`load_format` reads the JSON Schema from `AI/src/<NodeType>/format.json`. This schema is passed verbatim to Ollama's `format` field, which activates Ollama's constrained decoding to ensure responses match the schema.

`list_prompts` and `load_prompt` let users select from pre-written prompt files under each node type's `prompts/` directory.

In [ ]:
def load_format(node_type: str) -> dict:
    """
    Load the Ollama JSON Schema for a node type.

    The schema is stored in AI/src/<NodeType>/format.json and follows the
    Ollama JSON Schema format (compatible with Ollama's 'format' API field).

    Args:
        node_type: Capitalised node type name, e.g. 'Director'.

    Returns:
        Parsed JSON Schema dict.
    """
    path = os.path.join(BASE_DIR, node_type, "format.json")
    with open(path, "r") as f:
        return json.load(f)


def list_prompts(node_type: str) -> list[str]:
    """
    List all .txt prompt files in AI/src/<NodeType>/prompts/.

    Args:
        node_type: Capitalised node type name.

    Returns:
        List of .txt filenames (not full paths), empty list if the directory
        does not exist.
    """
    prompts_dir = os.path.join(BASE_DIR, node_type, "prompts")
    if not os.path.isdir(prompts_dir):
        return []
    return [f for f in os.listdir(prompts_dir) if f.endswith(".txt")]


def load_prompt(node_type: str, prompt_name: str) -> str:
    """
    Load a prompt from AI/src/<NodeType>/prompts/<prompt_name>.

    Args:
        node_type:   Capitalised node type name.
        prompt_name: Filename of the prompt, e.g. 'sample.txt'.

    Returns:
        Prompt text with leading/trailing whitespace stripped.
    """
    path = os.path.join(BASE_DIR, node_type, "prompts", prompt_name)
    with open(path, "r") as f:
        return f.read().strip()


# ── Preview: load and display the Director schema and sample prompt ────────────
fmt = load_format("Director")
print("Director schema required fields:", fmt.get("required"))
print("Director schema output properties:", list(fmt.get("properties", {}).keys()))

available_prompts = list_prompts("Director")
print(f"\nAvailable Director prompts: {available_prompts}")

if available_prompts:
    sample = load_prompt("Director", available_prompts[0])
    print(f"\n--- Sample prompt ({available_prompts[0]}) ---")
    print(sample)

### 1.3 Ollama Request Function

`send_request` wraps Ollama's `/api/generate` endpoint. Key points:
- The `format` field in the payload activates **constrained JSON decoding** — Ollama guarantees the response matches the schema.
- **Streaming mode** (`stream=True`) prints tokens as they arrive, useful for interactive use; non-streaming waits for the full response.
- Both modes return the same dict structure so callers handle them uniformly.

In [ ]:
def send_request(
    ollama_url: str,
    model: str,
    prompt: str,
    fmt: dict,
    stream: bool = False,
) -> dict:
    """
    Send a structured prompt to Ollama and return the parsed response.

    Args:
        ollama_url: Base URL of the Ollama server (e.g. 'http://localhost:11434').
        model:      Ollama model tag (e.g. 'llama3.1:8b' or 'nodpt').
        prompt:     The user prompt text.
        fmt:        JSON Schema dict — passed as Ollama's 'format' field to
                    constrain the output to valid structured JSON.
        stream:     If True, tokens are printed as they arrive.

    Returns:
        Dict with at least a 'response' key containing the model's reply.

    Raises:
        requests.exceptions.ConnectionError: if Ollama is not running.
        requests.exceptions.HTTPError: if the server returns a non-2xx status.
    """
    url = f"{ollama_url}/api/generate"
    payload = {
        "model":  model,
        "prompt": prompt,
        "stream": stream,
        "format": fmt,   # JSON Schema for constrained decoding
    }
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json=payload,
        stream=stream,
    )
    response.raise_for_status()

    if stream:
        # Print each token as it arrives, collect all tokens for return value
        collected = []
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                token = chunk.get("response", "")
                if token:
                    print(token, end="", flush=True)
                    collected.append(token)
                if chunk.get("done"):
                    break
        print()  # newline after streaming completes
        return {"response": "".join(collected)}

    return response.json()


print("send_request defined. Requires Ollama running at", DEFAULT_OLLAMA_URL)

### 1.4 Running a Prompt End-to-End

The following cell demonstrates a full request flow. Uncomment the call to `send_request` when Ollama is running.

**To start Ollama:**
```bash
ollama serve            # start the Ollama daemon
ollama pull llama3.1:8b # download the model if not already present
```

In [ ]:
def run_prompt(node_type_lower: str, prompt_text: str, model: str = DEFAULT_MODEL,
               ollama_url: str = DEFAULT_OLLAMA_URL, stream: bool = False):
    """
    High-level helper: load schema, send prompt, pretty-print the JSON response.

    Args:
        node_type_lower: Node type in lowercase (e.g. 'director').
        prompt_text:     The prompt string to send to the model.
        model:           Ollama model tag.
        ollama_url:      Ollama server base URL.
        stream:          Enable streaming mode.
    """
    node_type = node_type_lower.capitalize()
    fmt = load_format(node_type)
    print(f"Loaded schema for {node_type}")
    print(f"Model: {model}")
    print(f"Prompt: {prompt_text[:100]}{'...' if len(prompt_text) > 100 else ''}")
    print(f"Sending request to {ollama_url} ...\n")

    try:
        result = send_request(ollama_url, model, prompt_text, fmt, stream)
        response_text = result.get("response", "")
        try:
            parsed = json.loads(response_text)
            print("Response (parsed JSON):")
            print(json.dumps(parsed, indent=2))
        except json.JSONDecodeError:
            print("Response (raw):")
            print(response_text)
    except requests.exceptions.ConnectionError:
        print(f"Error: Could not connect to Ollama at {ollama_url}")
        print("Ensure Ollama is running: `ollama serve`")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")


# ── Example: director prompt from file ───────────────────────────────────────
director_prompt = load_prompt("Director", "sample.txt")
print("Sample director prompt loaded:")
print(director_prompt)
print("\n# Uncomment the next line to send to Ollama:")
# run_prompt("director", director_prompt)

### 1.5 Explore All Node Types

Loop through all four node types to inspect their schemas and sample prompts side-by-side — useful for understanding what each node type expects.

In [ ]:
for node_type in NODE_TYPES:
    fmt = load_format(node_type)
    prompts = list_prompts(node_type)

    print(f"\n{'='*60}")
    print(f"Node type: {node_type}")
    print(f"  Required output fields: {fmt.get('required')}")

    # Show the array field and its item fields
    props = fmt.get("properties", {})
    for key, value in props.items():
        if value.get("type") == "array":
            item_keys = list(value.get("items", {}).get("properties", {}).keys())
            print(f"  Array field '{key}' items: {item_keys}")

    # Show available prompts
    print(f"  Available prompts: {prompts}")

    if prompts:
        sample = load_prompt(node_type, prompts[0])
        preview = sample.split("\n")[0]  # First line only for brevity
        print(f"  Sample prompt first line: {preview}")

### 1.6 CLI Usage Reference

When executed as a script, `run.py` accepts the following arguments:

```bash
# List available prompts for a node type
python run.py director

# Use a prompt file
python run.py director --prompt sample.txt

# Use a custom inline prompt
python run.py agent --prompt-text "Write an authentication controller in Python"

# Use a custom model (e.g. after fine-tuning)
python run.py director --prompt sample.txt --model nodpt

# Enable streaming output
python run.py supervisor --prompt sample.txt --stream

# Custom Ollama URL (e.g. remote server)
python run.py manager --prompt sample.txt --ollama-url http://my-server:11434
```

---
## Part 2 — vLLM Sample Generator (`generate_vllm_samples.py`)

**Script:** `AI/src/generate_vllm_samples.py`

Generates **coding** and **book-writing** training samples via a [vLLM](https://github.com/vllm-project/vllm) server on port 8001. Unlike `generate_samples.py` (which targets NodPT node-type schemas), this script generates more general-purpose instruction-following data for broader fine-tuning.

### Key features
- **Randomised topics** — Large topic pools (30 coding domains × 12 languages, 20 book genres × 20 themes × 15 settings) ensure diverse, non-repetitive samples.
- **Async concurrency** — `aiohttp` + `asyncio.Semaphore` allows up to 8 concurrent requests by default.
- **Incremental output** — Appends to existing JSONL files; previously seen prompts are deduplicated.
- **Validation** — Each sample is checked for required fields (`prompt`, `response`) and minimum response length.

### 2.1 Imports and Configuration

In [ ]:
import argparse
import asyncio
import json
import os
import random
import sys
import time

import aiohttp

# ── Output directory ──────────────────────────────────────────────────────────
AI_SRC_DIR_VLLM = os.path.abspath("AI/src")
OUTPUT_DIR      = os.path.join(AI_SRC_DIR_VLLM, "vllm-samples")

# ── vLLM server defaults ──────────────────────────────────────────────────────
DEFAULT_ENDPOINT_VLLM  = "http://localhost:8001"                    # vLLM port
DEFAULT_MODEL_VLLM     = "meta-llama/Llama-3.1-70B-Instruct"
DEFAULT_CONCURRENCY_V  = 8   # Max simultaneous vLLM requests
DEFAULT_BATCH_SIZE_V   = 5   # Samples requested per API call

print("Configuration loaded.")
print(f"Output directory: {OUTPUT_DIR}")

### 2.2 Topic Pools

Large, diverse topic pools are key to generating varied training data. The random selection at prompt-build time ensures each batch contains unique combinations.

- **Coding samples** combine a domain topic + a programming language.
- **Book samples** combine a genre + a theme + a setting.

In [ ]:
# ── Coding topic pools ────────────────────────────────────────────────────────
CODING_TOPICS = [
    "web development", "mobile app development", "backend API",
    "database optimization", "DevOps automation", "machine learning",
    "data processing", "microservices", "cloud infrastructure",
    "security implementation", "testing framework", "CI/CD pipeline",
    "real-time analytics", "blockchain", "IoT systems",
    "game development", "desktop application", "CLI tools",
    "browser extensions", "REST API", "GraphQL API",
    "websocket server", "authentication system", "payment processing",
    "file processing", "image manipulation", "video streaming",
    "chatbot", "search engine", "recommendation system",
]

CODING_LANGUAGES = [
    "Python", "JavaScript", "TypeScript", "Go", "Rust",
    "Java", "C#", "Ruby", "PHP", "Swift", "Kotlin", "C++",
]

# ── Book writing topic pools ──────────────────────────────────────────────────
BOOK_GENRES = [
    "science fiction", "fantasy", "mystery", "thriller", "romance",
    "historical fiction", "horror", "adventure", "dystopian",
    "literary fiction", "young adult", "contemporary", "crime",
    "biography", "memoir", "self-help", "business", "technology",
    "philosophy", "psychology",
]

BOOK_THEMES = [
    "artificial intelligence", "time travel", "parallel universes",
    "post-apocalyptic survival", "space exploration", "ancient civilizations",
    "supernatural powers", "political intrigue", "family dynamics",
    "personal transformation", "social justice", "environmental crisis",
    "technological singularity", "human-robot relationships", "magical realism",
    "coming of age", "identity and belonging", "love and loss",
    "redemption", "revolution",
]

BOOK_SETTINGS = [
    "futuristic megacity", "medieval kingdom", "Victorian London",
    "modern Silicon Valley", "isolated space station", "underwater colony",
    "magical academy", "post-war Europe", "rural countryside",
    "cyberpunk metropolis", "ancient Rome", "distant planet",
    "parallel dimension", "small coastal town", "mountain monastery",
]

# ── Quick statistics ──────────────────────────────────────────────────────────
coding_combinations = len(CODING_TOPICS) * len(CODING_LANGUAGES)
book_combinations   = len(BOOK_GENRES) * len(BOOK_THEMES) * len(BOOK_SETTINGS)
print(f"Unique coding topic combinations:     {coding_combinations:,}")
print(f"Unique book topic combinations:       {book_combinations:,}")

### 2.3 Random Prompt Generators

These functions pick random elements from the topic pools and combine them using varied sentence templates. The template variation prevents the model from seeing the same framing repeatedly, which would reduce output diversity.

In [ ]:
def get_random_coding_prompt() -> str:
    """
    Generate a random coding task prompt string.

    Selects a random topic from CODING_TOPICS and a random language from
    CODING_LANGUAGES, then combines them using one of five sentence templates
    to maximise linguistic variety.

    Returns:
        A coding task prompt string, e.g.:
        'Implement a machine learning system in Python'
    """
    topic    = random.choice(CODING_TOPICS)
    language = random.choice(CODING_LANGUAGES)
    templates = [
        f"Implement a {topic} system in {language}",
        f"Create a {language} library for {topic}",
        f"Build a {topic} solution using {language}",
        f"Develop a {language} application for {topic}",
        f"Design and code a {topic} component in {language}",
    ]
    return random.choice(templates)


def get_random_book_prompt() -> str:
    """
    Generate a random book writing prompt string.

    Selects a random genre, theme, and setting, then combines them using one
    of five narrative prompt templates.

    Returns:
        A creative writing prompt, e.g.:
        'Write a fantasy story about time travel set in a medieval kingdom'
    """
    genre   = random.choice(BOOK_GENRES)
    theme   = random.choice(BOOK_THEMES)
    setting = random.choice(BOOK_SETTINGS)
    templates = [
        f"Write a {genre} story about {theme} set in a {setting}",
        f"Create a {genre} narrative exploring {theme} in a {setting}",
        f"Develop a {genre} plot centered on {theme} within a {setting}",
        f"Compose a {genre} tale featuring {theme} against the backdrop of a {setting}",
        f"Craft a {genre} storyline dealing with {theme} in a {setting}",
    ]
    return random.choice(templates)


# ── Generate and display a few example prompts ────────────────────────────────
print("Example coding prompts:")
for _ in range(5):
    print(f"  {get_random_coding_prompt()}")

print("\nExample book prompts:")
for _ in range(5):
    print(f"  {get_random_book_prompt()}")

### 2.4 Batch Prompt Builder

`build_generation_prompt` creates the meta-prompt that instructs the large model to produce multiple JSONL samples in one response. For coding samples it emphasises realistic code with error handling; for book samples it emphasises vivid prose with character and plot.

In [ ]:
def build_generation_prompt(
    sample_type: str,
    batch_size: int,
    existing_inputs: list | None = None,
) -> str:
    """
    Build the batch generation prompt sent to the vLLM-served large model.

    The prompt requests exactly `batch_size` JSONL lines. For coding samples,
    it instructs the model to include working code. For book samples, it asks
    for 500-1000 word story excerpts.

    Args:
        sample_type:     'coding' or 'book'.
        batch_size:      Number of JSONL lines to generate per API call.
        existing_inputs: Previously seen prompts to avoid duplicating.

    Returns:
        Formatted prompt string.
    """
    # Deduplication hint — cap at 20 to avoid bloating the prompt
    avoid_section = ""
    if existing_inputs:
        sample_list = existing_inputs[:20]
        avoid_section = (
            "\n\nDo NOT reuse these existing prompts (generate completely different ones):\n"
            + "\n".join(f"- {inp}" for inp in sample_list)
        )

    if sample_type == "coding":
        prompt = (
            f"Generate exactly {batch_size} diverse coding task samples in JSONL format.\n\n"
            "Each sample must be a JSON object on its own line with these fields:\n"
            '- "prompt": A specific coding task request (vary programming languages and domains)\n'
            '- "response": A detailed implementation including code, explanations, and best practices\n\n'
            "Requirements:\n"
            "1. Each prompt must be UNIQUE with different programming domains and languages\n"
            "2. Cover various areas: web, mobile, backend, data, DevOps, ML, security, etc.\n"
            "3. Include multiple programming languages: Python, JavaScript, TypeScript, Go, Java, C#, etc.\n"
            "4. Responses should contain realistic, working code with proper syntax\n"
            "5. Add explanations of the approach and key implementation details\n"
            "6. Include error handling and edge cases where appropriate"
            f"{avoid_section}\n\n"
            f"Output ONLY the {batch_size} JSONL lines, one per line. No markdown, no explanation, no code fences."
        )
    else:  # book writing
        prompt = (
            f"Generate exactly {batch_size} diverse book writing samples in JSONL format.\n\n"
            "Each sample must be a JSON object on its own line with these fields:\n"
            '- "prompt": A creative writing prompt with genre, theme, and setting\n'
            '- "response": A well-crafted story excerpt or chapter (500-1000 words)\n\n'
            "Requirements:\n"
            "1. Each prompt must be UNIQUE with different genres, themes, and settings\n"
            "2. Cover various genres: sci-fi, fantasy, mystery, thriller, literary fiction, etc.\n"
            "3. Include vivid descriptions, character development, and engaging dialogue\n"
            "4. Vary writing styles and narrative perspectives\n"
            "5. Create compelling plots with conflict and resolution\n"
            "6. Show rather than tell through sensory details and actions"
            f"{avoid_section}\n\n"
            f"Output ONLY the {batch_size} JSONL lines, one per line. No markdown, no explanation, no code fences."
        )

    return prompt


# ── Preview prompts ───────────────────────────────────────────────────────────
coding_gen_prompt = build_generation_prompt("coding", 3)
book_gen_prompt   = build_generation_prompt("book",   3)

print("=== Coding generation prompt (first 400 chars) ===")
print(coding_gen_prompt[:400])
print(f"\n... total length: {len(coding_gen_prompt)} chars")

print("\n=== Book generation prompt (first 400 chars) ===")
print(book_gen_prompt[:400])

### 2.5 vLLM API Call

`call_vllm` sends a single async request to the vLLM OpenAI-compatible endpoint. It uses the same error-handling pattern as `call_tensorrt` in `generate_samples.py` — returning `(content, None)` on success and `(None, error_string)` on failure — so the caller can log errors without raising exceptions mid-generation.

In [ ]:
async def call_vllm(
    session: aiohttp.ClientSession,
    endpoint: str,
    model: str,
    prompt: str,
    temperature: float = 0.8,
):
    """
    Send a chat-completion request to the vLLM OpenAI-compatible endpoint.

    vLLM (on port 8001 by default) exposes the same /v1/chat/completions API
    as OpenAI. This function uses it to generate JSONL training lines.

    Args:
        session:     Shared aiohttp session for connection pooling.
        endpoint:    vLLM server base URL (e.g. 'http://localhost:8001').
        model:       Model ID served by vLLM.
        prompt:      User-facing batch generation prompt.
        temperature: Sampling temperature (0.8 balances diversity vs. coherence).

    Returns:
        Tuple (response_text: str | None, error: str | None).
    """
    url = f"{endpoint}/v1/chat/completions"
    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a training data generator. You produce JSONL lines "
                    "for fine-tuning language models. Output ONLY valid JSONL — "
                    "one JSON object per line, no markdown, no extra text."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": 4096,
        "stream": False,
    }

    try:
        async with session.post(
            url,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=aiohttp.ClientTimeout(total=300),
        ) as resp:
            if resp.status != 200:
                text = await resp.text()
                return None, f"HTTP {resp.status}: {text[:200]}"
            data = await resp.json()
            content = data["choices"][0]["message"]["content"]
            return content, None
    except asyncio.TimeoutError:
        return None, "request timed out (300s)"
    except aiohttp.ClientError as e:
        return None, f"connection error: {e}"


print("call_vllm defined. Requires a running vLLM server at", DEFAULT_ENDPOINT_VLLM)

### 2.6 Response Parser

Validates each generated JSONL line against a minimal schema:
- Must be valid JSON.
- Must contain both `prompt` and `response` string fields.
- Neither field may be empty.
- Response must be at least 50 characters (too-short responses are not useful training data).
- Markdown code-fence lines (```` ``` ````) are silently skipped.

In [ ]:
def parse_generated_lines_vllm(raw_text: str, sample_type: str):
    """
    Parse raw model output into validated JSONL samples.

    Validation criteria:
      - Each line must be parseable JSON.
      - Must contain 'prompt' (str, non-empty) and 'response' (str, non-empty).
      - Response must be at least 50 characters.
      - Markdown code-fence lines are discarded.

    Args:
        raw_text:    Multi-line string from the vLLM response.
        sample_type: 'coding' or 'book' (not used for validation, kept for symmetry).

    Returns:
        Tuple (valid_samples: list[dict], invalid_count: int).
    """
    valid_samples = []
    invalid_count = 0

    for line in raw_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("```"):
            continue  # Ignore markdown code-fence markers

        try:
            sample = json.loads(line)
        except json.JSONDecodeError:
            invalid_count += 1
            continue

        # Must have both required fields
        if not all(k in sample for k in ("prompt", "response")):
            invalid_count += 1
            continue

        if not isinstance(sample["prompt"], str) or not sample["prompt"].strip():
            invalid_count += 1
            continue

        if not isinstance(sample["response"], str) or not sample["response"].strip():
            invalid_count += 1
            continue

        # Reject suspiciously short responses
        if len(sample["response"]) < 50:
            invalid_count += 1
            continue

        valid_samples.append(sample)

    return valid_samples, invalid_count


# ── Demo validation ───────────────────────────────────────────────────────────
demo_raw = """{
    "prompt": "Implement a REST API in Python",
    "response": "Here is a Flask REST API implementation with full CRUD operations and error handling..."
}""".replace("\n", "")
valid, invalid = parse_generated_lines_vllm(demo_raw + "\n{\"bad\": true}", "coding")
print(f"Demo: {len(valid)} valid, {invalid} invalid")

### 2.7 Orchestration — Generate All vLLM Samples

The orchestration loop follows the same pattern as `generate_samples.py`:
1. Load existing JSONL to build a deduplication set.
2. Launch batches concurrently (capped by `asyncio.Semaphore`).
3. Stop early if no progress is made (prevents spinning on a poorly-configured server).
4. Append all new valid samples to the output file.

In [ ]:
async def generate_batch_vllm(
    session, endpoint, model, sample_type, batch_size, existing_inputs, temperature=0.8
):
    """Generate one batch and return (valid_list, invalid_count, error_or_none)."""
    prompt = build_generation_prompt(sample_type, batch_size, existing_inputs)
    raw, err = await call_vllm(session, endpoint, model, prompt, temperature)
    if err:
        return [], 0, err
    valid, invalid = parse_generated_lines_vllm(raw, sample_type)
    return valid, invalid, None


async def generate_all_vllm(args):
    """
    Orchestrate concurrent batch generation for coding and/or book sample types.

    Args:
        args: Namespace with fields:
              type, count, endpoint, model, concurrency, batch_size, temperature.
    """
    sample_types = ["coding", "book"] if args.type == "all" else [args.type]
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for sample_type in sample_types:
        print(f"\n{'='*60}")
        print(f"Generating {args.count} '{sample_type}' samples")
        print(f"Endpoint: {args.endpoint} | Model: {args.model}")
        print(f"{'='*60}")

        output_path = os.path.join(OUTPUT_DIR, f"{sample_type}.jsonl")

        # Load existing prompts for deduplication
        existing_inputs = set()
        if os.path.isfile(output_path):
            with open(output_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            existing_inputs.add(json.loads(line).get("prompt", ""))
                        except json.JSONDecodeError:
                            pass
            print(f"Existing samples: {len(existing_inputs)}")

        collected     = []
        total_invalid = 0
        start_time    = time.time()
        remaining     = args.count
        semaphore     = asyncio.Semaphore(args.concurrency)

        async with aiohttp.ClientSession() as session:
            while remaining > 0:
                num_batches = min(
                    args.concurrency,
                    (remaining + args.batch_size - 1) // args.batch_size,
                )
                batch_sizes = [
                    min(args.batch_size, remaining - i * args.batch_size)
                    for i in range(num_batches)
                    if min(args.batch_size, remaining - i * args.batch_size) > 0
                ]

                async def bounded(bs):
                    async with semaphore:
                        return await generate_batch_vllm(
                            session, args.endpoint, args.model,
                            sample_type, bs, list(existing_inputs), args.temperature,
                        )

                results    = await asyncio.gather(*[bounded(s) for s in batch_sizes])
                prev_count = len(collected)

                for valid, invalid, err in results:
                    if err:
                        print(f"  Batch error: {err}")
                        continue
                    total_invalid += invalid
                    for sample in valid:
                        if sample["prompt"] not in existing_inputs:
                            collected.append(sample)
                            existing_inputs.add(sample["prompt"])

                remaining = args.count - len(collected)
                elapsed   = time.time() - start_time
                print(f"  Progress: {len(collected)}/{args.count} ({total_invalid} invalid) [{elapsed:.1f}s]")

                if remaining <= 0:
                    break
                # Stop if no progress at all
                if len(collected) == prev_count:
                    print("  No new samples generated. Stopping.")
                    break

        if collected:
            with open(output_path, "a", encoding="utf-8") as f:
                for sample in collected:
                    f.write(json.dumps(sample, ensure_ascii=False) + "\n")
            elapsed = time.time() - start_time
            print(f"\nSaved {len(collected)} samples → {output_path} ({elapsed:.1f}s)")
        else:
            print(f"\nNo valid samples generated for {sample_type}.")

    print(f"\n{'='*60}\nDone. Samples saved to {OUTPUT_DIR}/\n{'='*60}")


print("generate_all_vllm defined. Example usage (requires vLLM server):")
print("  import types")
print("  args = types.SimpleNamespace(type='coding', count=20, endpoint='http://localhost:8001',")
print("         model='meta-llama/Llama-3.1-70B-Instruct', concurrency=8, batch_size=5, temperature=0.8)")
print("  asyncio.run(generate_all_vllm(args))")

### 2.8 CLI Usage Reference

```bash
# From AI/src/

# Generate 10 coding samples
python generate_vllm_samples.py --type coding --count 10

# Generate 20 book writing samples
python generate_vllm_samples.py --type book --count 20

# Generate both types (50 each)
python generate_vllm_samples.py --type all --count 50

# Custom endpoint and model
python generate_vllm_samples.py --type all --count 30 \
    --endpoint http://localhost:8001 \
    --model meta-llama/Llama-3.1-70B-Instruct
```

Generated samples are saved to `AI/src/vllm-samples/coding.jsonl` and `AI/src/vllm-samples/book.jsonl`. See `AI/src/VLLM_SAMPLES_GUIDE.md` for detailed guidance.

---

## Summary

| Script | API endpoint | Output format | Use case |
|--------|-------------|---------------|----------|
| `run.py` | Ollama `/api/generate` | Constrained JSON (via `format` field) | Interactive inference with NodPT node types |
| `generate_vllm_samples.py` | vLLM `/v1/chat/completions` | JSONL `{prompt, response}` | Bulk training data for coding/book models |